In [1]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                              bidirectional=True, divide_output=True, pscan=True, use_cuda=False)
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [3]:
class WholeBrainPatchEmbed3D(nn.Module):
    def __init__(self, brain_size=120, patch_size=8, d_model=32):
        super().__init__()
        self.patch_size = patch_size
        self.grid_size = brain_size // patch_size
        self.n_tokens = self.grid_size ** 3
        self.d_model = d_model

        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)

        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, volume):
        tokens = self.patch_conv(volume).flatten(2).transpose(1, 2)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, :, :]
        occupancy = F.max_pool3d((volume.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool()
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid


class WholeBrainBranch(nn.Module):
    def __init__(self, brain_size=120, patch_size=8, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        self.patch_embed = WholeBrainPatchEmbed3D(brain_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)

    def forward(self, volume):
        tokens, valid = self.patch_embed(volume)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)
        return pooled


class WholeBrainModel(nn.Module):
    """Single-modality -- use for MRI-only or PET-only."""
    def __init__(self, brain_size=120, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, volume):
        pooled = self.branch(volume)
        return self.classifier(self.dropout(pooled))


class MultimodalWholeBrainModel(nn.Module):
    """Late fusion -- separate MRI/PET whole-brain branches, concatenated
    before the classifier. Mirrors existing ROI-based multimodal model."""
    def __init__(self, brain_size=120, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = WholeBrainBranch(brain_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_vol, pet_vol):
        mri_pooled = self.mri_branch(mri_vol)
        pet_pooled = self.pet_branch(pet_vol)
        fused = torch.cat([mri_pooled, pet_pooled], dim=1)
        return self.classifier(self.dropout(fused))

In [4]:
COHORT_CSV       = "D:/mamba_model/thesis_cohort_final.csv"
WB_MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_mri_aug"
WB_PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_wholebrain_pet_aug"
CKPT_DIR         = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

BRAIN_SIZE = 120  # must match TARGET_SIZE from extracted data

df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [5]:
class WholeBrainDataset(Dataset):
    """Single-modality (MRI or PET) whole-brain dataset."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        vol = np.array(np.load(f"{self.cache_dir}/{key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(vol).unsqueeze(0), torch.tensor(label, dtype=torch.long), key


class MultimodalWholeBrainDataset(Dataset):
    """Pairs MRI and PET whole-brain volumes for the same subject/seed."""
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir, self.pet_cache_dir = mri_cache_dir, pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_vol = np.array(np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        pet_vol = np.array(np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return (torch.from_numpy(mri_vol).unsqueeze(0), torch.from_numpy(pet_vol).unsqueeze(0),
                torch.tensor(label, dtype=torch.long), mri_key)

In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for vol, labels, _ in loader:
        vol, labels = vol.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(vol), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for vol, labels, _ in loader:
            vol, labels = vol.to(device), labels.to(device)
            out = model(vol)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_vol, pet_vol, labels, _ in loader:
        mri_vol, pet_vol, labels = mri_vol.to(device), pet_vol.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(mri_vol, pet_vol), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for mri_vol, pet_vol, labels, _ in loader:
            mri_vol, pet_vol, labels = mri_vol.to(device), pet_vol.to(device), labels.to(device)
            out = model(mri_vol, pet_vol)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [7]:
def measure_inference_time(model, loader, device, is_multimodal, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            if is_multimodal:
                mri_vol, pet_vol, labels, _ = batch
                mri_vol, pet_vol = mri_vol.to(device), pet_vol.to(device)
                bs = mri_vol.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(mri_vol, pet_vol)
            else:
                vol, labels, _ = batch
                vol = vol.to(device)
                bs = vol.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(vol)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / bs)
    return np.mean(times), np.std(times)

def try_compute_flops(model, loader, device, is_multimodal):
    try:
        model.eval()
        batch = next(iter(loader))
        with torch.no_grad():
            if is_multimodal:
                mri_vol, pet_vol, labels, _ = batch
                inputs = (mri_vol[:1].to(device), pet_vol[:1].to(device))
            else:
                vol, labels, _ = batch
                inputs = (vol[:1].to(device),)
            macs, _ = profile(model, inputs=inputs, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})")
        return None

def run_one_seed(seed, model_class, train_loader, val_loader, test_loader, is_multimodal, save_prefix, max_epochs=101, patience=15):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = model_class(brain_size=BRAIN_SIZE, d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    train_fn = train_epoch_mm if is_multimodal else train_epoch
    eval_fn = evaluate_mm if is_multimodal else evaluate

    best_val_loss, no_improve, best_epoch, total_time = float("inf"), 0, 0, 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = eval_fn(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_time += epoch_time
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:
            best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = eval_fn(model, test_loader, criterion, device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_mean, inf_std = measure_inference_time(model, test_loader, device, is_multimodal)
    flops = try_compute_flops(model, test_loader, device, is_multimodal)

    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_time/60:.1f}min | inf={inf_mean*1000:.2f}ms | {f'{flops/1e9:.2f}GFLOPs' if flops else 'N/A'}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch,
            "train_time_sec": total_time, "n_params": n_params, "inf_time_ms": inf_mean * 1000, "flops": flops}

In [8]:
# MRI ONLY

BATCH_SIZE = 4
wb_mri_train_loader = DataLoader(WholeBrainDataset(X_train, y_train, WB_MRI_CACHE_AUG, True, True), batch_size=BATCH_SIZE, shuffle=True)
wb_mri_val_loader   = DataLoader(WholeBrainDataset(X_val, y_val, WB_MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)
wb_mri_test_loader  = DataLoader(WholeBrainDataset(X_test, y_test, WB_MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== WHOLE-BRAIN MRI-ONLY (token-matched): 3-seed run ===")
wb_mri_results = [run_one_seed(s, WholeBrainModel, wb_mri_train_loader, wb_mri_val_loader, wb_mri_test_loader,
                                False, "vim_wholebrain_mri") for s in [1, 7, 123]]

=== WHOLE-BRAIN MRI-ONLY (token-matched): 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.6979 |     0.6921 |   0.5000 |   0.9524 |   0.0476 |  56.8s
     2 |     0.6972 |     0.6914 |   0.4762 |   0.1429 |   0.8095 |  10.8s
     3 |     0.6939 |     0.6910 |   0.5000 |   0.9524 |   0.0476 |  10.7s
     4 |     0.6904 |     0.6906 |   0.5238 |   0.1429 |   0.9048 |  10.6s
     5 |     0.6879 |     0.6920 |   0.5238 |   0.0952 |   0.9524 |  10.9s
     6 |     0.6857 |     0.6893 |   0.5476 |   0.7619 |   0.3333 |  11.0s
     7 |     0.6894 |     0.6890 |   0.5476 |   0.7143 |   0.3810 |  10.7s
     8 |     0.6840 |     0.6894 |   0.5714 |   0.4762 |   0.6667 |  10.6s
     9 |     0.6824 |     0.6886 |   0.5952 |   0.6190 |   0.5714 |  10.5s
    10 |     0.6768 |     0.6926 |   0.5000 |   0.9524 |   0.0476 |  10.8s
    11 |     0.6823 |     0.688

In [9]:
# PET ONLY

wb_pet_train_loader = DataLoader(WholeBrainDataset(X_train, y_train, WB_PET_CACHE_AUG, False, True), batch_size=BATCH_SIZE, shuffle=True)
wb_pet_val_loader   = DataLoader(WholeBrainDataset(X_val, y_val, WB_PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)
wb_pet_test_loader  = DataLoader(WholeBrainDataset(X_test, y_test, WB_PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== WHOLE-BRAIN PET-ONLY (token-matched): 3-seed run ===")
wb_pet_results = [run_one_seed(s, WholeBrainModel, wb_pet_train_loader, wb_pet_val_loader, wb_pet_test_loader,
                                False, "vim_wholebrain_pet") for s in [1, 7, 123]]

=== WHOLE-BRAIN PET-ONLY (token-matched): 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7082 |     0.6924 |   0.4762 |   0.9524 |   0.0000 |  60.7s
     2 |     0.7044 |     0.6914 |   0.5000 |   0.0000 |   1.0000 |  10.7s
     3 |     0.6975 |     0.6909 |   0.4762 |   0.9524 |   0.0000 |  10.5s
     4 |     0.6937 |     0.6912 |   0.5000 |   0.0000 |   1.0000 |  10.5s
     5 |     0.6899 |     0.6915 |   0.5000 |   0.0000 |   1.0000 |  10.5s
     6 |     0.6898 |     0.6868 |   0.6429 |   0.7143 |   0.5714 |  10.5s
     7 |     0.6956 |     0.6851 |   0.6429 |   0.5238 |   0.7619 |  10.5s
     8 |     0.6862 |     0.6851 |   0.5000 |   0.0000 |   1.0000 |  10.6s
     9 |     0.6871 |     0.6826 |   0.5238 |   0.0476 |   1.0000 |  10.5s
    10 |     0.6804 |     0.6819 |   0.5476 |   0.9048 |   0.1905 |  10.5s
    11 |     0.6823 |     0.677

In [10]:
# MULTIMODALITY (MRI & PET)

wb_mm_train_loader = DataLoader(MultimodalWholeBrainDataset(X_train, y_train, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, True), batch_size=4, shuffle=True)
wb_mm_val_loader   = DataLoader(MultimodalWholeBrainDataset(X_val, y_val, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, False), batch_size=4, shuffle=False)
wb_mm_test_loader  = DataLoader(MultimodalWholeBrainDataset(X_test, y_test, WB_MRI_CACHE_AUG, WB_PET_CACHE_AUG, False), batch_size=4, shuffle=False)

print("=== WHOLE-BRAIN MULTIMODAL (token-matched): 3-seed run ===")
wb_mm_results = [run_one_seed(s, MultimodalWholeBrainModel, wb_mm_train_loader, wb_mm_val_loader, wb_mm_test_loader,
                               True, "vim_wholebrain_mm") for s in [1, 7, 123]]

=== WHOLE-BRAIN MULTIMODAL (token-matched): 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7055 |     0.6976 |   0.5000 |   1.0000 |   0.0000 | 173.1s
     2 |     0.7075 |     0.6975 |   0.5000 |   1.0000 |   0.0000 | 171.8s
     3 |     0.6901 |     0.6914 |   0.5714 |   0.3810 |   0.7619 | 172.9s
     4 |     0.7029 |     0.6911 |   0.4762 |   0.0476 |   0.9048 | 173.3s
     5 |     0.6842 |     0.6952 |   0.5000 |   1.0000 |   0.0000 | 170.4s
     6 |     0.6881 |     0.6902 |   0.5238 |   1.0000 |   0.0476 | 171.6s
     7 |     0.6771 |     0.6879 |   0.4762 |   0.1429 |   0.8095 | 173.3s
     8 |     0.6808 |     0.6900 |   0.5238 |   1.0000 |   0.0476 | 170.2s
     9 |     0.6865 |     0.6899 |   0.5238 |   1.0000 |   0.0476 | 171.8s
    10 |     0.6827 |     0.6885 |   0.5238 |   1.0000 |   0.0476 | 171.7s
    11 |     0.6747 |     0.6

In [11]:
def summarize(results, name):
    accs, tprs, tnrs = [r["acc"] for r in results], [r["tpr"] for r in results], [r["tnr"] for r in results]
    print(f"{name}: Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
          f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
          f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}%")

print("=== Whole-brain (token-matched) ===")
summarize(wb_mri_results, "MRI-only")
summarize(wb_pet_results, "PET-only")
summarize(wb_mm_results, "Multimodal")

print("\n=== Compare to ROI-based (no stats) ===")
print("MRI-only:   Acc=68.3±2.7% | TPR=63.5±7.3% | TNR=73.0±2.7%")
print("PET-only:   Acc=62.7±5.0% | TPR=68.3±12.0% | TNR=57.1±4.8%")
print("Multimodal: Acc=65.1±1.4% | TPR=58.7±5.5% | TNR=71.4±4.8%")

=== Whole-brain (token-matched) ===
MRI-only: Acc=56.3±2.7% | TPR=65.1±7.3% | TNR=47.6±12.6%
PET-only: Acc=63.5±1.4% | TPR=52.4±0.0% | TNR=74.6±2.7%
Multimodal: Acc=63.5±1.4% | TPR=55.6±2.7% | TNR=71.4±0.0%

=== Compare to ROI-based (no stats) ===
MRI-only:   Acc=68.3±2.7% | TPR=63.5±7.3% | TNR=73.0±2.7%
PET-only:   Acc=62.7±5.0% | TPR=68.3±12.0% | TNR=57.1±4.8%
Multimodal: Acc=65.1±1.4% | TPR=58.7±5.5% | TNR=71.4±4.8%
